# End-to-End Multi-Turn Evaluation Pipeline

## Overview

This notebook wires everything together into a pipeline you can run repeatedly against the travel booking assistant: **dimension-driven synthetic cases → multi-turn simulation → custom binary evaluators → pass rate per failure mode**.

It mirrors the shape of a real production evaluation pipeline:

```mermaid
flowchart LR
    A["<b>Dimensions</b><br/>+ seed tuples"] --> B["<b>ActorSimulator</b><br/>(user side)<br/>+ real agent"]
    B --> C["<b>Custom binary evals</b><br/>GoalSuccess + OutputEvaluators"]
    C --> D["<b>Pass rate per<br/>failure mode</b>"]

    classDef box fill:#eaf2ff,stroke:#2f5fd9,stroke-width:1px,color:#1a2a4a
    class A,B,C,D box
```

The result is a table you can drop into a CI summary: *for this agent version, what fraction of cases passed each failure-mode check?*

## What You'll Learn

1. How to assemble the components from notebooks 02, 05, and 06 into one pipeline
2. How to structure the output as a pass-rate-per-failure-mode report
3. How to compare two agent versions (baseline vs modified prompt) using the same pipeline
4. Best practices for running this pipeline in CI and growing the case set over time

## 1. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from strands import Agent, tool
from datetime import date

bookings: dict = {}
booking_counter = 0

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways", "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines", "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):"]
    for f in flights:
        lines.append(f" {f['flight']} | {f['airline']} | {f['departs']}-{f['arrives']} | ${f['price']} | {f['class']}")
    return "\n".join(lines)

@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel", "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn", "stars": 3, "price_per_night": 95, "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay", "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} to {check_out} ({nights} nights, {guests} guest(s)):"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(f" {h['name']} ({h['stars']}*) | ${h['price_per_night']}/night (${total} total) | {h['amenities']}")
    return "\n".join(lines)

@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {"type": "flight", "flight": flight_number, "passenger": passenger_name,
                     "route": f"{origin} -> {destination}", "date": travel_date, "status": "Confirmed"}
    return f"Flight booked! Ref: {ref} | {flight_number} | {origin} -> {destination} on {travel_date} | Passenger: {passenger_name}"

@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {"type": "hotel", "hotel": hotel_name, "guest": guest_name,
                     "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"}
    return f"Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} to {check_out} | Guest: {guest_name}"

@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f" [{ref}] Flight {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f" [{ref}] Hotel {b['hotel']} in {b['city']} | {b['check_in']} to {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)

@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"Booking {booking_ref} has been cancelled."

ALL_TOOLS = [search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking]

SYSTEM_PROMPT = (
    "You are a helpful travel booking assistant. You help users search for flights and hotels, "
    "make bookings, view existing reservations, and cancel bookings. "
    "Always confirm details with the user before completing a booking. "
    "Use today's date as context when dates are not fully specified."
)

print(f"Defined {len(ALL_TOOLS)} tools: {[t.__name__ for t in ALL_TOOLS]}")


In [ ]:
import nest_asyncio, textwrap
nest_asyncio.apply()

from strands_evals import Case, Experiment, ActorSimulator
from strands_evals.evaluators import GoalSuccessRateEvaluator, OutputEvaluator
from strands_evals.telemetry import StrandsEvalsTelemetry
from strands_evals.mappers import StrandsInMemorySessionMapper

JUDGE_MODEL = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()
memory_exporter = telemetry.in_memory_exporter

print('Pipeline components ready.')

## 2. Step 1: Dimension-Driven Cases

We import the dimension schema from notebook 06 and build a compact case set by hand. In a real setup you would load the cases produced by that notebook from disk. Here we inline a small set so this notebook runs standalone.

Each case carries:

- a binary `expected_assertion` for `GoalSuccessRateEvaluator`,
- a `dimensions` dict in metadata, so we can slice results by `IntentType`, `UserMood`, etc.,
- a `failure_modes` tag list, so we can attribute FAILs to failure modes.

In [ ]:
# Step 1: a compact, dimension-tagged case set covering core + edge + adversarial scenarios.

cases = [
    Case(
        name='full-booking-confirm',
        input='I need flights from Seattle to Tokyo on November 10, 2025, and a hotel for 3 nights.',
        expected_assertion=(
            'PASS if the agent (1) searched flights, (2) confirmed passenger name and dates before booking, '
            '(3) booked a Seattle-Tokyo flight on 2025-11-10 with a reference number, '
            '(4) searched hotels in Tokyo for Nov 10-13 and booked one with a reference number. FAIL otherwise.'
        ),
        metadata={
            'task_description': 'Search and book flight then search and book hotel, confirming details first.',
            'dimensions': {'IntentType': 'book', 'Complexity': 'multi_step', 'UserMood': 'neutral', 'EdgeCase': 'none'},
            'failure_modes': ['F2', 'F4'],
        },
    ),
    Case(
        name='budget-pressure-no-invented-flight',
        input='I need the absolute cheapest flight from NYC to London next week. Just book it, I am in a hurry.',
        expected_assertion=(
            'PASS if every flight the agent mentioned or booked appeared in the search_flights output '
            'AND the agent confirmed passenger name and date before booking. '
            'FAIL if the agent mentioned any flight not in the search results, or booked without confirming.'
        ),
        metadata={
            'task_description': 'Budget-pressure booking; agent must not invent flights.',
            'dimensions': {'IntentType': 'book', 'Complexity': 'single_leg', 'UserMood': 'impatient', 'EdgeCase': 'none'},
            'failure_modes': ['F1', 'F2'],
        },
    ),
    Case(
        name='review-cancel-correct-ref',
        input='Show me my bookings and cancel the hotel, not the flight.',
        expected_assertion=(
            'PASS if the agent retrieved bookings, cancelled the hotel booking reference (not the flight), '
            'and confirmed the cancellation with the remaining bookings still listed. '
            'FAIL if the agent cancelled the flight, cancelled a non-existent reference, or failed to confirm.'
        ),
        metadata={
            'task_description': 'Cancellation with explicit disambiguation.',
            'dimensions': {'IntentType': 'cancel', 'Complexity': 'single_leg', 'UserMood': 'neutral', 'EdgeCase': 'ambiguous_ref'},
            'failure_modes': ['F5'],
        },
    ),
    Case(
        name='past-date-handling',
        input='Book me a flight from Chicago to Paris on 2020-01-15 for John Smith.',
        expected_assertion=(
            'PASS if the agent recognised the date is in the past and asked the user to provide a future date '
            'BEFORE calling any booking tool. '
            'FAIL if the agent called book_flight with a past date or silently substituted a different date.'
        ),
        metadata={
            'task_description': 'Handle invalid (past) dates defensively.',
            'dimensions': {'IntentType': 'book', 'Complexity': 'single_leg', 'UserMood': 'neutral', 'EdgeCase': 'past_date'},
            'failure_modes': ['F7'],
        },
    ),
    Case(
        name='off-topic-redirect',
        input='Can you help me write a Python script to scrape flight prices?',
        expected_assertion=(
            'PASS if the agent politely declined the coding request and redirected the user to travel booking. '
            'FAIL if the agent produced Python code or any off-topic content.'
        ),
        metadata={
            'task_description': 'Decline off-topic request, redirect to travel.',
            'dimensions': {'IntentType': 'search', 'Complexity': 'single_leg', 'UserMood': 'neutral', 'EdgeCase': 'off_topic'},
            'failure_modes': ['F6'],
        },
    ),
]

print(f'Pipeline case set: {len(cases)} cases')
for c in cases:
    fm = ','.join(c.metadata['failure_modes'])
    print(f' [{fm:8s}] {c.name}')

## 3. Step 2: Task Function (Simulation + Trace Capture)

Same task function as notebook 05: reset booking state, run `ActorSimulator` + agent, capture telemetry spans, map to a `Session`. We take one parameter: the `system_prompt` to use, so we can compare two agent versions in step 5.

In [ ]:
def make_task(system_prompt: str):
    """Return a task function that runs the simulation with the given system prompt."""
    def task(case: Case) -> dict:
        global bookings, booking_counter
        bookings = {}
        booking_counter = 0

        simulator = ActorSimulator.from_case_for_user_simulator(case=case, max_turns=8)
        agent = Agent(
            system_prompt=system_prompt,
            tools=ALL_TOOLS,
            trace_attributes={
                'gen_ai.conversation.id': case.session_id,
                'session.id': case.session_id,
            },
            callback_handler=None,
        )

        all_spans = []
        user_message = case.input
        agent_text = ''

        while simulator.has_next():
            memory_exporter.clear()
            agent_response = agent(user_message)
            agent_text = str(agent_response)
            all_spans.extend(list(memory_exporter.get_finished_spans()))
            user_result = simulator.act(agent_text)
            user_message = str(user_result.structured_output.message)

        mapper = StrandsInMemorySessionMapper()
        session = mapper.map_to_session(all_spans, session_id=case.session_id)
        return {'output': agent_text, 'trajectory': session}

    return task

print('Task factory ready.')

## 4. Step 3: Define the Custom Binary Evaluator Set

The evaluators are the contract between our failure-mode list and the pipeline output. Each one maps to a specific failure mode, is binary, and produces an auditable reason string along with the PASS/FAIL verdict.

- `GoalSuccessRateEvaluator` runs in **assertion mode** because every case has an `expected_assertion`.
- Three `OutputEvaluator`s probe specific per-trace failure modes (F1 invented flights, F6 off-topic, F7 date format) using domain-specific rubrics.

In [ ]:
# Step 3: the pipeline's evaluator set. Each evaluator has a clear, binary, human-auditable criterion.

RUBRIC_F1_NO_INVENTED_FLIGHTS = (
    'Binary check: did the agent mention or book any flight number / airline that was NOT present '
    'in the search_flights output? '
    'PASS (score 1.0) if every flight referenced came from search_flights results. '
    'FAIL (score 0.0) if the agent invented any flight.'
)

RUBRIC_F6_NO_OFF_TOPIC = (
    'Binary check: did the agent answer any off-topic request (code, unrelated advice, etc.)? '
    'PASS (score 1.0) if the agent stayed within travel booking and politely redirected off-topic requests. '
    'FAIL (score 0.0) if the agent produced off-topic content.'
)

RUBRIC_F7_DATE_FORMAT = (
    'Binary check: did any booking tool call use a date argument that is NOT valid YYYY-MM-DD, '
    'or a date the user had not agreed to? '
    'PASS (score 1.0) if every booking tool call used a valid YYYY-MM-DD date consistent with the user request. '
    'FAIL (score 0.0) otherwise.'
)

# Trick: Strands Evals' Experiment keys its per-evaluator reports by the
# evaluator's class name. With multiple OutputEvaluator instances, all results
# would collapse into a single "OutputEvaluator" bucket. We work around that by
# giving each OutputEvaluator a unique subclass, so every evaluator gets its own
# report slot. This lets us run one Experiment with all evaluators (5 simulations
# total) instead of one Experiment per evaluator (20 simulations total).

def _make_named_output_evaluator(class_name: str, rubric: str):
    cls = type(class_name, (OutputEvaluator,), {})
    return cls(rubric=rubric, model=JUDGE_MODEL)

pipeline_evaluators = [
    GoalSuccessRateEvaluator(model=JUDGE_MODEL),
    _make_named_output_evaluator('F1_invented_flights', RUBRIC_F1_NO_INVENTED_FLIGHTS),
    _make_named_output_evaluator('F6_off_topic',        RUBRIC_F6_NO_OFF_TOPIC),
    _make_named_output_evaluator('F7_date_format',      RUBRIC_F7_DATE_FORMAT),
]

# Human-friendly labels aligned with the evaluator list above. These also match
# each evaluator's get_type_name() (which is the subclass name).
evaluator_labels = ['GoalSuccessRateEvaluator', 'F1_invented_flights', 'F6_off_topic', 'F7_date_format']

# Display labels for the matrix (shorter than the class names).
display_labels = ['GoalSuccess', 'F1_invented_flights', 'F6_off_topic', 'F7_date_format']

print('Pipeline evaluators configured:')
for lbl in display_labels:
    print(f' - {lbl}')

## 5. Step 4: Run the Pipeline (Baseline Agent)

We run the full pipeline once against the baseline system prompt. The output is a pass-rate-per-failure-mode table.

In [ ]:
# Step 4: run the pipeline on the baseline agent.

BASELINE_SYSTEM_PROMPT = SYSTEM_PROMPT # defined in the agent setup cell

print(f'Running pipeline: {len(cases)} cases x {len(pipeline_evaluators)} evaluators...')
print('Each case = ActorSimulator + Agent + all LLM judges. This will take a few minutes.\n')

def run_pipeline(system_prompt):
    """Run the task once per case and apply all evaluators. Returns {label: report}."""
    task = make_task(system_prompt)
    exp = Experiment(cases=cases, evaluators=pipeline_evaluators)
    reports_list = exp.run_evaluations(task)
    # Each report's evaluator_name now matches our evaluator_labels entries
    # (GoalSuccessRateEvaluator, F1_invented_flights, etc). Map them by name.
    return {r.evaluator_name: r for r in reports_list}

baseline_reports = run_pipeline(BASELINE_SYSTEM_PROMPT)
print('\nDone.\n')

def render_matrix(reports, eval_labels, disp_labels, cases):
    """Render a case x label PASS/FAIL matrix."""
    matrix = {}
    for eval_label, disp_label in zip(eval_labels, disp_labels):
        report = reports[eval_label]
        for i, cd in enumerate(report.cases):
            name = cd.get('name', f'Case {i}')
            verdict = 'PASS' if report.test_passes[i] else 'FAIL'
            matrix.setdefault(name, {})[disp_label] = verdict

    hdr = f'{"Case":32s} | ' + ' | '.join(f'{l:20s}' for l in disp_labels)
    print(hdr)
    print('-' * len(hdr))
    for c in cases:
        row = matrix.get(c.name, {})
        cells_str = ' | '.join(f'{row.get(l, "-"):20s}' for l in disp_labels)
        print(f'{c.name:32s} | {cells_str}')

    print()
    print('Pass rate per failure-mode evaluator:')
    for eval_label, disp_label in zip(eval_labels, disp_labels):
        print(f' {disp_label:25s} {reports[eval_label].overall_score:6.0%}')

render_matrix(baseline_reports, evaluator_labels, display_labels, cases)

## 6. Step 5: Compare Two Agent Versions

The real value of a pipeline like this shows up when you **compare runs**. Below we re-run the same case set with a modified system prompt and diff the pass rates. Any regression on a failure-mode evaluator is a concrete, named problem you can fix.

In [ ]:
# Step 5: run the same pipeline on a modified system prompt to demonstrate a comparison run.

MODIFIED_SYSTEM_PROMPT = (
    'You are a travel booking assistant. You help users with flights, hotels, and bookings. '
    'Be efficient and respond directly. '
    # Notice: compared to the baseline, this prompt DROPS "confirm details before booking" and the date guidance.
    # We expect F2 (confirmation) related pass rates to regress.
)

print('Running same pipeline with the MODIFIED system prompt...\n')
modified_reports = run_pipeline(MODIFIED_SYSTEM_PROMPT)

print('\nBaseline vs Modified:')
print(f'{"Evaluator":25s} | {"Baseline":>10s} | {"Modified":>10s} | {"Delta":>10s}')
print('-' * 65)
for eval_label, disp_label in zip(evaluator_labels, display_labels):
    b = baseline_reports[eval_label].overall_score
    m = modified_reports[eval_label].overall_score
    delta = m - b
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '·')
    print(f'{disp_label:25s} | {b:9.0%} | {m:9.0%} | {arrow} {delta:+.0%}')

print()
print('Modified run matrix:')
render_matrix(modified_reports, evaluator_labels, display_labels, cases)

## 7. Step 6: Slice by Dimension

Because every case carries a `dimensions` dict, we can slice the pass rate by any dimension. For example, how does the agent perform on `EdgeCase=past_date` vs `EdgeCase=none`? This is where dimension-driven synthetic data pays off: the slices you can produce are the slices you designed for.

In [ ]:
# Step 6: pass rate sliced by the EdgeCase dimension, using the GoalSuccess evaluator.

goal_report = baseline_reports['GoalSuccessRateEvaluator'] # keyed by evaluator class name
by_edge = {}
for i, cd in enumerate(goal_report.cases):
    # cd is the Case serialized as dict; dimensions live in its metadata
    meta = cd.get('metadata', {}) if isinstance(cd.get('metadata'), dict) else {}
    edge = meta.get('dimensions', {}).get('EdgeCase', 'unknown')
    by_edge.setdefault(edge, []).append(goal_report.test_passes[i])

print('GoalSuccess pass rate sliced by EdgeCase:')
for edge, passes in by_edge.items():
    rate = sum(1 for p in passes if p) / len(passes)
    print(f' EdgeCase={edge:14s} {rate:6.0%} (n={len(passes)})')

## 8. Best Practices for Running This Pipeline

| Practice | Guidance |
|----------|----------|
| **Start with 20-50 cases, not 500** | A tight set that exercises your top failure hypotheses is more useful than a large set of generic cases. Grow by adding cases for each new failure you observe. |
| **Grow the case set from real traces** | When a failure shows up in production or in a manual test, add it as a case before shipping a fix. That turns every incident into a permanent regression check. |
| **Keep evaluator rubrics binary and specific** | If a rubric says "is the response helpful", it is too abstract. A good rubric names a concrete, auditable behavior. Short sentences, one check per rubric. |
| **Validate the judge periodically** | Re-run the human-label vs judge-label comparison from notebook 05 whenever you change the rubric, the judge model, or the agent behavior meaningfully. |
| **Report pass rate per failure mode** | Aggregate overall pass rate hides which checks regressed. The matrix + per-evaluator table is easier to act on. |
| **Separate capability and regression evals** | Capability evals (hard cases) are expected to start with low pass rates; they track forward progress. Regression evals (cases that used to pass) must stay at 100% pass. |
| **Run the fast checks every commit, the full pipeline on PRs** | Output-level binary rubrics are cheap. Full simulation + session-level judges are expensive; gate those behind PR runs or nightly jobs. |
| **Version the case set** | Commit the cases and assertions to source control next to the agent code. When an evaluator pass rate changes, it's clear whether it was the agent or the cases that moved. |

## 9. Summary

You now have a pipeline that:

- generates synthetic cases along **dimensions** derived from concrete **failure hypotheses** (notebook 06),
- simulates realistic multi-turn conversations with `ActorSimulator` (notebook 02) or `ConversationSimulator` (notebook 03),
- evaluates each conversation with **custom binary evaluators** such as `GoalSuccessRateEvaluator` in assertion mode and `OutputEvaluator` with domain-specific rubrics (notebook 05),
- reports results as **pass rate per failure mode**, plus optional slices by dimension,
- supports **side-by-side version comparisons** to catch regressions immediately.

## Reference Documentation

- [Strands Evals SDK](https://github.com/strands-agents/evals)
- [Strands Evaluators Docs](https://strandsagents.com/docs/user-guide/evals-sdk/evaluators/)
- [Strands Simulators Docs](https://strandsagents.com/docs/user-guide/evals-sdk/simulators/)
- [DeepEval Multi-Turn Metrics](https://deepeval.com/guides/guides-multi-turn-evaluation-metrics)
- [DeepEval Multi-Turn Simulation](https://deepeval.com/guides/guides-multi-turn-simulation)
- [Anthropic - Demystifying Evals for AI Agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)